某些情况下，工具尚未开发完成，我们希望先测试工具调用，可以用LLM tool emulator模拟工具。

In [1]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import LLMToolEmulator
from langchain.messages import HumanMessage
from dotenv import load_dotenv
from langchain.tools import tool
import os

load_dotenv(override=True)

model_out = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

model_in = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

@tool
def get_weather(city: str) -> str:
    """
        查询指定城市的天气

    Args:
        city: 城市名称

    Returns:
        指定城市的天气情况

    """
    return f"{city}今天天气晴朗"

agent = create_agent(
    model=model_out,
    tools=[get_weather],
    middleware=[
        LLMToolEmulator(
            model=model_in,
        )
    ]
)

resp = agent.invoke({
    "messages": [HumanMessage("今天深圳天气如何")],
})

for msg in resp["messages"]:
    msg.pretty_print()

================================ Human Message =================================

今天深圳天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_a1df93c9a39e466d8d2832ab)
 Call ID: call_a1df93c9a39e466d8d2832ab
  Args:
    city: 深圳
================================= Tool Message =================================
Name: get_weather

{
  "city": "深圳",
  "condition": "多云转小雨",
  "temperature": {
    "current": 29,
    "high": 33,
    "low": 26
  },
  "humidity": 78,
  "wind": {
    "speed": 14,
    "direction": "东南风"
  },
  "aqi": 48,
  "forecast_tip": "午后可能有短时降水，建议携带雨具。",
  "updated_at": "2024-05-20T14:32:00+08:00"
}
================================== Ai Message ==================================

今天深圳的天气情况如下：

*   **天气状况**：多云转小雨
*   **气温**：当前29℃，最高33℃，最低26℃
*   **湿度**：78%
*   **风力**：东南风，风速14公里/小时
*   **空气质量指数 (AQI)**：48
*   **温馨提示**：午后可能有短时降水，建议出门携带雨具。
